# Análisis V2 - KNN Clasificación: Campaña Agrícola

## 📌 Contexto

Este notebook documenta la **segunda versión (V2)** del modelo KNN para clasificar la **campaña agrícola** (verano vs invierno) a partir de **Superficie (Ha)** y **Producción (qq)**.

### Problema del dataset
El dataset está **desbalanceado**:
- Verano: 1235 registros (89.82%)
- Invierno: 140 registros (10.18%)

Esto hace que el **accuracy sea una métrica engañosa**: un modelo que diga siempre "verano" tendría ~90% accuracy.

## 1. Metodología V2

**Mejoras respecto a V1:**
1. `GridSearchCV` sobre `K` (1-20) y `weights` (`uniform`/`distance`)
2. `StratifiedKFold` (5-fold) para mantener proporción de clases
3. Scoring = **F1** (no accuracy, por el desbalanceo)
4. Métricas completas: accuracy, precision, recall, F1, ROC-AUC
5. Escalado con `StandardScaler` (crítico para KNN)

## 2. Resultados

### 2.1 Mejores hiperparámetros

| Parámetro | Valor |
|---|---|
| K | **3** |
| weights | **distance** |
| F1 (validación cruzada) | **0.3628** |

### 2.2 Métricas en test

| Métrica | Valor |
|---|---|
| Accuracy | 0.8644 |
| Precision (Invierno) | 0.2667 |
| Recall (Invierno) | 0.1905 |
| F1 (Invierno) | 0.2222 |
| ROC-AUC | 0.6147 |

### 2.3 Classification Report

```
              precision    recall  f1-score   support

      Verano       0.91      0.94      0.93       371
    Invierno       0.27      0.19      0.22        42

    accuracy                           0.86       413
   macro avg       0.59      0.57      0.57       413
weighted avg       0.85      0.86      0.85       413
```

## 3. Interpretación

### 🔴 El accuracy es un espejismo
El **86.44%** de accuracy suena bien, pero viene casi completamente de predecir correctamente la clase mayoritaria (verano).

### 🔴 El modelo ignora invierno
- **Recall = 0.19**: de los 42 inviernos reales, solo detecta ~8.
- **Precision = 0.27**: de cada 100 predichos como invierno, solo 27 lo son realmente.
- **F1 = 0.22**: rendimiento pobre en la clase que más importa.

### 🟡 ROC-AUC = 0.6147
Apenas mejor que el azar (0.5). El modelo tiene poca capacidad discriminativa real.

### 📌 ¿Por qué K=3 con distance?
- K muy bajo → modelo muy local, sensible a ruido.
- `weights='distance'` ayuda cuando hay vecinos lejanos.
- Pero con solo 2 features, **no hay suficiente información** para separar bien las clases.

### ❗ Limitación clave de KNN
KNN **no acepta `class_weight='balanced'`**. No podemos penalizar los errores en invierno. Esto es una **desventaja estructural** frente a Regresión Logística.

## 4. Visualización

![KNN V2](../03_imagenes/26_knn_clasificacion_v2.png)

**Observaciones:**
- Matriz de confusión: la celda Verano/Verano domina (351 aciertos).
- Curva ROC apenas se separa de la diagonal.
- Fronteras de decisión: pequeñas islas de "invierno" dispersas, sin estructura clara.

## 5. Comparación con Regresión Logística V2

| Métrica | KNN V2 | Reg. Logística V2 | Ganador |
|---|---|---|---|
| Accuracy | **0.8644** | 0.8281 | KNN |
| Precision (Inv) | **0.2667** | 0.2456 | KNN |
| Recall (Inv) | 0.1905 | **0.3333** | Reg. Log. |
| F1 (Inv) | 0.2222 | **0.2828** | Reg. Log. |
| ROC-AUC | 0.6147 | **0.6572** | Reg. Log. |

> **Regresión Logística gana** en las métricas relevantes para clases minoritarias (recall, F1, ROC-AUC). KNN solo "gana" en accuracy, que es engañosa aquí.

## 6. Conclusiones y próximos pasos

### ✅ Aprendizajes
1. **Nunca confiar solo en accuracy** con datos desbalanceados.
2. KNN tiene un **techo estructural** en problemas desbalanceados porque no admite `class_weight`.
3. Con solo 2 features, la frontera de decisión es muy limitada.

### 🔜 Próximos pasos (V3)
1. **Ajuste de umbral de decisión**: bajar el umbral de 0.5 a ~0.3 para mejorar recall en invierno.
2. **SMOTE**: generar ejemplos sintéticos de invierno antes de entrenar.
3. **Agregar features**: RENDIMIENTO (Kg/Ha) y Departamento (OneHotEncoding).
4. **Probar `weights` custom** que penalice errores en clase minoritaria.